### 1. Project Setup and Model Verification

In this section, we verify that the trained SARIMAX model from Task 5 is available in the project directory and ready for integration into the Django application.


In [4]:
from pathlib import Path

# Get the current project directory
PROJECT_DIR = Path.cwd()

# Model file located directly inside the project folder
MODEL_PATH = PROJECT_DIR / "sarimax_weather_model.pkl"

# Verify the model file
print("Project directory:", PROJECT_DIR)
print("Model file exists:", MODEL_PATH.exists())

if MODEL_PATH.exists():
    print(f"Model size: {MODEL_PATH.stat().st_size / (1024 ** 2):.2f} MB")
else:
    print("Model not found. Check the notebook's working directory.")

Project directory: d:\cellula_tasks\Weather Forecast Prediction\Task_6_Weather_web
Model file exists: True
Model size: 5.16 MB


### 2. Load and Verify the Trained SARIMAX Model

In this section, we load the serialized SARIMAX model trained in Task 5 and verify that it can be restored successfully. This allows us to reuse the trained model in the Django application without retraining it.


In [5]:
import sys
import site

user_site = site.getusersitepackages()

if user_site not in sys.path:
    sys.path.append(user_site)

print("User site added:", user_site)
print("Path available:", user_site in sys.path)

User site added: C:\Users\moham\AppData\Roaming\Python\Python311\site-packages
Path available: True


In [6]:
import statsmodels

print("Statsmodels version:", statsmodels.__version__)

Statsmodels version: 0.15.0


In [7]:
import pickle
from pathlib import Path

MODEL_PATH = Path.cwd() / "sarimax_weather_model.pkl"

with open(MODEL_PATH, "rb") as file:
    weather_model = pickle.load(file)

print("Model loaded successfully!")
print("Model type:", type(weather_model).__name__)
print("Training observations:", weather_model.nobs)
print("Model parameters:", weather_model.param_names)

Model loaded successfully!
Model type: SARIMAXResultsWrapper
Training observations: 2922
Model parameters: ['intercept', 'sin_1', 'cos_1', 'sin_2', 'cos_2', 'sin_3', 'cos_3', 'ar.L1', 'ma.L1', 'sigma2']


## 1. Testing the Saved Weather Forecast Model

In this section, we test the loaded SARIMAX model by generating a short-term weather forecast.

The model uses Fourier features to represent yearly seasonality and predicts daily average temperature (`T2M`) based on the learned time-series patterns.


In [9]:
import numpy as np
import pandas as pd

# Generate dates immediately after the training period
forecast_steps = 7

forecast_dates = pd.date_range(
    start="2024-01-01",
    periods=forecast_steps,
    freq="D"
)

# Generate Fourier features using the same period and K used during training
day_of_year = forecast_dates.day_of_year.to_numpy()
fourier_features = {}

for k in range(1, 4):
    fourier_features[f"sin_{k}"] = np.sin(
        2 * np.pi * k * day_of_year / 365.25
    )
    fourier_features[f"cos_{k}"] = np.cos(
        2 * np.pi * k * day_of_year / 365.25
    )

forecast_exog = pd.DataFrame(
    fourier_features,
    index=forecast_dates
)

# Generate predictions
forecast_result = weather_model.get_forecast(
    steps=forecast_steps,
    exog=forecast_exog
)

forecast_df = pd.DataFrame({
    "Date": forecast_dates,
    "Predicted_Temperature_C": forecast_result.predicted_mean.to_numpy()
})

forecast_df

,Date,Predicted_Temperature_C
0,2024-01-01,14.374933
1,2024-01-02,13.701668
2,2024-01-03,13.239553
3,2024-01-04,12.917710
4,2024-01-05,12.689490
5,2024-01-06,12.524185
6,2024-01-07,12.401588


## 2. Reproducing the Training Features

In this section, we recreate the Fourier feature generation function used during model training. Using the same feature names, seasonal period, and mathematical formulas helps ensure consistency between the training notebook and the deployed application.


In [10]:
import numpy as np
import pandas as pd

def fourier_features(index, K=3, period=365.25):
    doy = index.day_of_year.to_numpy()

    feats = {}

    for k in range(1, K + 1):
        feats[f"sin_{k}"] = np.sin(2 * np.pi * k * doy / period)
        feats[f"cos_{k}"] = np.cos(2 * np.pi * k * doy / period)

    return pd.DataFrame(feats, index=index)


forecast_steps = 7

forecast_dates = pd.date_range(
    start="2024-01-01",
    periods=forecast_steps,
    freq="D"
)

forecast_exog = fourier_features(
    forecast_dates,
    K=3,
    period=365.25
)

forecast_result = weather_model.get_forecast(
    steps=forecast_steps,
    exog=forecast_exog
)

forecast_df = pd.DataFrame({
    "Date": forecast_dates,
    "Predicted_Temperature_C": forecast_result.predicted_mean.to_numpy()
})

display(forecast_df)

,Date,Predicted_Temperature_C
0,2024-01-01,14.374933
1,2024-01-02,13.701668
2,2024-01-03,13.239553
3,2024-01-04,12.917710
4,2024-01-05,12.689490
5,2024-01-06,12.524185
6,2024-01-07,12.401588
